# Datagen

Generate realistic Delta tables and a Direct Lake semantic model from a Power BI `.vpax` file.

## Setup

1. **Attach a Lakehouse** — click the Lakehouse icon in the left sidebar and select (or create) a Lakehouse
2. **Upload your `.vpax` file** — place it in `Files/datagen/` in the attached Lakehouse
   - Export a `.vpax` from [DAX Studio](https://daxstudio.org/) → Advanced → Export Metrics
3. **Run Cell 1** below to install datagen (auto-downloads from GitHub if needed)
4. **Edit Cell 2** — change the `.vpax` filename to match yours
5. **Run Cell 2** — generates Delta tables, deploys a semantic model, and prints a comparison report


In [ ]:
# Install datagen — downloads the latest release from GitHub if not cached locally
import glob, subprocess, sys, os, urllib.request, json

if not os.path.isdir('/lakehouse/default'):
    raise RuntimeError(
        'No lakehouse attached. Click the Lakehouse icon in the left sidebar, '
        'then select or create a Lakehouse before running this notebook.'
    )

DATAGEN_DIR = "/lakehouse/default/Files/datagen"
os.makedirs(DATAGEN_DIR, exist_ok=True)

USE_LATEST = True  # Set False to use a cached .whl instead of downloading

def _get_latest_whl():
    api_url = "https://api.github.com/repos/dbrownems/datagen/releases/latest"
    with urllib.request.urlopen(api_url) as resp:
        release = json.loads(resp.read())
    asset = next(a for a in release["assets"] if a["name"].endswith(".whl"))
    local = f"{DATAGEN_DIR}/{asset['name']}"
    if not os.path.exists(local):
        print(f"Downloading {asset['name']} ...")
        urllib.request.urlretrieve(asset["browser_download_url"], local)
    return local

if USE_LATEST:
    whl = _get_latest_whl()
else:
    whls = sorted(glob.glob(f"{DATAGEN_DIR}/datagen_fabric-*.whl"))
    whl = whls[-1] if whls else _get_latest_whl()

print(f"Installing {os.path.basename(whl)}")
subprocess.check_call([
    sys.executable, "-m", "pip", "install", whl, "semantic-link-labs",
    "-q", "--no-warn-conflicts", "--disable-pip-version-check", "--force-reinstall", "--no-deps",
])
subprocess.check_call([sys.executable, "-m", "pip", "install", "semantic-link-labs", "-q", "--no-warn-conflicts", "--disable-pip-version-check"])
import datagen; print(f"Ready — datagen v{datagen.__version__}")

# Verify the installed version matches the loaded version
import importlib.metadata
installed = importlib.metadata.version('datagen-fabric')
loaded = datagen.__version__
if installed != loaded:
    raise RuntimeError(
        f'Version mismatch: installed v{installed} but loaded v{loaded}. '
        f'Please restart the Spark session (Session → Stop session) and re-run.'
    )


## Generate

Edit the `.vpax` filename below, then run the cell. This will:

1. **Parse** the `.vpax` and infer data distributions from column statistics
2. **Generate** Delta tables matching the original row counts and cardinality
3. **Deploy** a Direct Lake semantic model with all measures, relationships, and column metadata
4. **Compare** the generated tables against the expected statistics and print a report

### Options

| Parameter | Default | Description |
|---|---|---|
| `mode` | `"direct_lake"` | `"import"` for Power Query import mode |
| `deploy_model` | `True` | `False` to skip semantic model deployment |
| `compare` | `True` | `False` to skip comparison report |
| `overwrite_tables` | `True` | `False` to skip existing tables, only generate missing |
| `overwrite_model` | `True` | `False` to fail if semantic model already exists |
| `seed` | `42` | Random seed for reproducible generation |


In [ ]:
from datagen import generate

VPAX_FILE = ""  # ← set your .vpax filename, or leave blank to auto-detect

# Auto-detect: use the first .vpax file in the datagen folder
if not VPAX_FILE:
    vpax_files = sorted(glob.glob(f"{DATAGEN_DIR}/*.vpax"))
    if not vpax_files:
        raise FileNotFoundError(f"No .vpax files found in {DATAGEN_DIR}. Upload one and re-run.")
    vpax_path = vpax_files[0]
    print(f"Auto-detected: {os.path.basename(vpax_path)}")
else:
    vpax_path = f"{DATAGEN_DIR}/{VPAX_FILE}"

report = generate(
    spark,
    vpax_path,
    compare=False,
    overwrite_tables=False,
    overwrite_model=True,
)